In [1]:
import os
import h5py
import numpy as np
from Utils import *
from MAPCsim import *
from TrafficGenerator import TrafficGenerator
# from TrafficGenerator import TrafficGenerator, poisson_fixed_events, generate_burst_traffic, generate_vr_traffic

# RL Model (e.g., PPO)
from CustomEnv import * # my Custom environment
from stable_baselines3 import PPO, TD3, DQN, A2C
from stable_baselines3.a2c import MlpPolicy
from stable_baselines3.common.vec_env import DummyVecEnv


# From example notebook
import gymnasium as gym
import matplotlib.pyplot as plt
from stable_baselines3.common.monitor import Monitor
from stable_baselines3.common.results_plotter import load_results, ts2xy
from stable_baselines3.common.noise import NormalActionNoise
from stable_baselines3.common.callbacks import BaseCallback, EvalCallback

###### Input parameters
validation_flag = 'no'


traffic_type = 'Bursty'
traffic_load = 'high'



# Scenario-related
AP_NUMBER = 4
STA_NUMBER = 16
GRID_VALUE = 60
SCENARIO_TYPE = 'grid'

sim = '30metros-16STAs'
walls = np.array([[0, GRID_VALUE, GRID_VALUE/2, GRID_VALUE/2], 
                  [GRID_VALUE/2, GRID_VALUE/2, 0, GRID_VALUE]])

# System-related parameters
TXOP_DURATION = 5E-3
PN_DBM = -95
CCA = -82
BW = 80
NSS = 2
L = 12E3

ITERATIONS = 1

### Channel-related parameters
MaxTxPower, NSC = TXpowerCalc(BW, NSS)


# ### Load deployment data
h5file_deployments_path = os.path.join('/home/david/Documents/Papers/journal_ML_CSR/python Code/deployments datasets', sim, 'deployment_datasets.h5')
with h5py.File(h5file_deployments_path, 'r') as f:
    STA_matrix_save = f['STA_matrix_save'][:]
    channelMatrix_save = f['channelMatrix_save'][:]
    RSSI_dB_vector_to_export_save = f['RSSI_dB_vector_to_export_save'][:]

### Output directory    
output_dir = os.path.join(os.getcwd(), 'Results/Simulation')

iter = 0
EDCAaccessCategory = 'BE'
# Check if the traffic type is valid
if EDCAaccessCategory is None:
    raise ValueError(f"Invalid traffic type: {traffic_type}. Valid types are 'Poisson', 'Bursty', 'VR'.")


### Deployment-dependent data
AP_matrix, STA_matrix = AP_STA_coordinates(AP_NUMBER, STA_NUMBER, SCENARIO_TYPE, GRID_VALUE)
# STA_matrix = STA_matrix_save[:, :, iter]

# Association
association = AP_STA_Association(AP_NUMBER, STA_NUMBER, SCENARIO_TYPE)

# Plot deployment
# PlotDeployment(AP_matrix, STA_matrix, association, GRID_VALUE, walls)

# Channel matrix. Uncomment if using pre-saved channel matrix.  
channelMatrix = channelMatrix_save[:, :, iter]

# RSSI vector. Uncomment if using pre-saved RSSI vector.
RSSI_dB_vector_to_export = RSSI_dB_vector_to_export_save[:, :, iter]

# Compute the channelMatrix and RSSI_dB_vector_to_export if they aren't provided as pre-saved datasets
# channelMatrix, RSSI_dB_vector_to_export = GetChannelMatrix(MaxTxPower, CCA, AP_matrix, STA_matrix, SCENARIO_TYPE, walls, checkSegmentIntersection, Getloss)

# Compute the overheads
preTX_overheadsDCF, preTX_overheadsCSR, DCFoverheads, CSRoverheads = OverheadsCalc(EDCAaccessCategory)

CGs_STAs, TxPowerMatrix = CG_creationTPC(AP_NUMBER, STA_NUMBER, PN_DBM, NSC, NSS, 
                                            association, channelMatrix, MaxTxPower, 
                                            CG_filter='on', TPC_method=None)    # TPC Optimization method: None, 'PSO', 'IPOPT', 'DE'
                                            

per_STA_DCF_throughput_bianchi = Throughput_DCF_bianchi(AP_NUMBER, STA_NUMBER, association, RSSI_dB_vector_to_export, PN_DBM, NSC, NSS, TXOP_DURATION, 
                                                        DCFoverheads, EDCAaccessCategory)


# Simulation duration
timestamp_to_stop = 1  # seconds

# Set the seed
seed = 1

# Simulation Configuration
sim_config = {
    'AP_NUMBER': AP_NUMBER,
    'STA_NUMBER': STA_NUMBER,
    'association': association,
    'MaxTxPower': MaxTxPower,
    'channelMatrix': channelMatrix,
    'traffic_type': traffic_type,
    'validation_flag': 'no',
    'TXOP_DURATION': TXOP_DURATION,
    'PN_DBM': PN_DBM,
    'NSS': NSS,
    'NSC': NSC,
    'preTX_overheadsDCF': preTX_overheadsDCF,
    'preTX_overheadsCSR': preTX_overheadsCSR,
    'DCFoverheads': DCFoverheads,
    'CSRoverheads': CSRoverheads,
    'timestamp_to_stop': timestamp_to_stop,
    'CGs_STAs': CGs_STAs,
    'TxPowerMatrix': TxPowerMatrix
}

In [ ]:
#########################################################################
# Set the seed
seed = 1

timestamp_to_stop = 1

# Create a Gym-compatible environment
def create_env():
    np.random.seed(seed)
    simulator = MAPCsim(sim_config)  # new "MAPC simulator" object
    simulator.simulation_system = 'CSR'
    simulator.CGs_STAs = CGs_STAs
    simulator.TxPowerMatrix = TxPowerMatrix
    simulator.accessCategory = EDCAaccessCategory
    return CustomEnv(sim_config, simulator)


# Create log dir
log_dir = '/home/david/Documents/Papers/journal_ML_CSR/python Code/trained_models'
os.makedirs(log_dir, exist_ok=True)

# Create and wrap the environment
env = create_env()  # Create the environment
env = Monitor(env, log_dir)  # Wrap with Monitor
# env = DummyVecEnv([lambda: env])  # Wrap with DummyVecEnv
# callback = SaveOnBestTrainingRewardCallback(check_freq=1000, log_dir=log_dir)

# Use deterministic actions for evaluation
callback = EvalCallback(env, best_model_save_path=log_dir,
                             log_path=log_dir, eval_freq=500,
                             deterministic=False, render=False)

# Initialize PPO agent
model = PPO("MultiInputPolicy", env, verbose=0)

num_episodes = 100000  # Number of episodes to train
total_timesteps_per_episode = 20000  # Number of timesteps per episode as max, 
                                    # the actual number may be less and it depends on the truncated flag, which is True when: 
                                    # truncated = bool(self.simulator.sim_timeline >= self.simulator.timestamp_to_stop)

# Training loop with custom traffic and episodes
for episode in range(num_episodes):
    # Generate new traffic for this episode
    STAs_arrivals_matrix = TrafficGenerator(
        STA_NUMBER, validation_flag, traffic_type, traffic_load, L, per_STA_DCF_throughput_bianchi, 
        EVENT_NUMBER = 15000 # Number of events considered for traffic generation
    )

    # Validate that the traffic lasts more than timestamp_to_stop
    if any(x < timestamp_to_stop for x in [STAs_arrivals_matrix[i][-1] for i in range(STA_NUMBER)]):
        raise ValueError(f'Traffic should last more than timestamp_to_stop: {timestamp_to_stop} seconds') 
    
    # Set traffic for the simulator and environment
    simulator = env.env.simulator  # Access the simulator from the vectorized env
    simulator.STA_queue_timeline = STAs_arrivals_matrix
    model.learn(total_timesteps=total_timesteps_per_episode, callback=callback)

    # Reset the environment for the next episode
    env.reset() 
